# 01 — Night limits (−18° and −12°) per `day_obs`

Ephemeris only, no data and no judgement. Two definitions carried side by side (handoff §1): the −18° → −18° night is the
operational night used for the loss accounting; the −12° → −12° night is reported alongside for comparison with the
expected-visits model in `rubin_nights` (which uses −12°, confirmed from its source). Output: `data/step0/night_limits.csv`.

In [ ]:
import sys, logging
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, "..")
from time_loss import DAY_OBS_MIN, DAY_OBS_MAX, GAP_MIN_MINUTES, GAP_MIN_SENSITIVITY, DAY_OBS_WEATHER_BIT_START
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 300); pd.set_option("display.max_colwidth", 80)
logging.basicConfig(level=logging.WARNING)
for name in ("httpx", "numexpr", "time_loss.access"):
    logging.getLogger(name).setLevel(logging.WARNING)
STEP0 = Path("../data/step0"); CACHE = Path("../data/cache"); PROC = Path("../data/processed")
for p in (STEP0, CACHE, PROC): p.mkdir(parents=True, exist_ok=True)

In [ ]:
from time_loss import nights
limits = nights.night_limits(DAY_OBS_MIN, DAY_OBS_MAX)
limits.to_csv(STEP0 / "night_limits.csv", index=False)
print(len(limits), "nights", limits.day_obs.min(), "->", limits.day_obs.max())
limits[["night_hours_18", "night_hours_12"]].agg(["min", "median", "max", "sum"]).round(2)

In [ ]:
import matplotlib.pyplot as plt
from time_loss.style import COLORS
fig, ax = plt.subplots(figsize=(9, 3))
x = pd.to_datetime(limits.day_obs.astype(str))
ax.plot(x, limits.night_hours_12, color=COLORS["grid"], lw=2, label="−12° → −12°")
ax.plot(x, limits.night_hours_18, color=COLORS["ink"], lw=2, label="−18° → −18°")
ax.set_ylabel("night length [h]"); ax.legend(frameon=False); ax.grid(alpha=.3)
fig.tight_layout()